# Foubert AI — Exploratory Data Analysis
**Dataset:** 3 days — 30 april, 1 mei (feestdag), 2 mei 2026  
**Goal:** Identify key signals for predicting where/when to sell ice cream and how much to bring.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ── Style ──────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#F8F7F4',
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'axes.spines.left': False,
    'axes.spines.bottom': False,
    'axes.grid':        True,
    'grid.color':       'white',
    'grid.linewidth':   1.2,
    'font.family':      'sans-serif',
    'font.size':        11,
    'axes.titlesize':   13,
    'axes.titleweight': 'bold',
    'axes.labelsize':   11,
    'xtick.bottom':     False,
    'ytick.left':       False,
})

BLUE   = '#3266AD'
GRAY   = '#73726C'
AMBER  = '#D07A2A'
GREEN  = '#1D9E75'
TEAL   = '#5DCAA5'
RED    = '#E24B4A'

DAYS   = ['30 apr\n(weekday)', '1 mei\n(holiday)', '2 mei\n(saturday)']
print('Libraries loaded.')

## 1 — Load your real data
Replace the summary dict below with actual TSV reads once you have the files.

In [ ]:
# ── Summary stats from README (replace with real data when available) ──────
summary = pd.DataFrame({
    'day':          ['30 apr (do)', '1 mei (feest)', '2 mei (za)'],
    'day_type':     ['weekday', 'holiday', 'weekend'],
    'vans':         [15, 15, 13],
    'sales':        [616, 996, 607],
    'sale_items':   [1815, 3047, 2411],
    'reservations': [5, 11, 22],
    'calls':        [363, 968, 435],
    'gps_points':   [229716, 203501, 237150],
})
summary['basket_size']   = (summary['sale_items'] / summary['sales']).round(2)
summary['calls_per_van'] = (summary['calls']      / summary['vans']).round(1)

# ── Load actual TSV files like this ───────────────────────────────────────
# BASE = 'foubertai_export'          # adjust to your folder
# sales_30 = pd.read_csv(f'{BASE}/2026-04-30/02_sales.tsv', sep='\t', parse_dates=['datetime_start','datetime_stop'])
# sales_01 = pd.read_csv(f'{BASE}/2026-05-01/02_sales.tsv', sep='\t', parse_dates=['datetime_start','datetime_stop'])
# sales_02 = pd.read_csv(f'{BASE}/2026-05-02/02_sales.tsv', sep='\t', parse_dates=['datetime_start','datetime_stop'])
# sales_all = pd.concat([sales_30, sales_01, sales_02], ignore_index=True)
# sales_all['date'] = sales_all['datetime_start'].dt.date
# sales_all['hour'] = sales_all['datetime_start'].dt.hour

summary

## 2 — KPI overview

In [ ]:
kpis = [
    ('Total sales (3 days)',   summary['sales'].sum(),          ''),
    ('Peak day sales',         summary['sales'].max(),          '1 mei'),
    ('Total calls',            summary['calls'].sum(),          ''),
    ('Calls unmet (May 2)',    '~69 %',                          '302 of 435'),
    ('Total reservations',     summary['reservations'].sum(),   ''),
    ('Max basket size',        summary['basket_size'].max(),    '2 mei (za)'),
]

fig, axes = plt.subplots(1, len(kpis), figsize=(16, 2.2))
for ax, (label, val, sub) in zip(axes, kpis):
    ax.set_axis_off()
    ax.set_facecolor('#F0EEE8')
    fig.patches  # just iterate
    rect = mpatches.FancyBboxPatch((0.05, 0.05), 0.9, 0.9,
        boxstyle='round,pad=0.02', linewidth=0,
        facecolor='#F0EEE8', transform=ax.transAxes, clip_on=False)
    ax.add_patch(rect)
    ax.text(0.5, 0.72, str(val), ha='center', va='center',
            fontsize=22, fontweight='bold', color='#1a1a1a', transform=ax.transAxes)
    ax.text(0.5, 0.38, label, ha='center', va='center',
            fontsize=9.5, color='#666', transform=ax.transAxes)
    if sub:
        ax.text(0.5, 0.16, sub, ha='center', va='center',
                fontsize=8.5, color='#999', transform=ax.transAxes)

fig.suptitle('3-day KPI overview', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 3 — Sales, calls & reservations per day (absolute + indexed)

In [ ]:
x   = np.arange(3)
w   = 0.25

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, indexed, title in zip(
    axes,
    [False, True],
    ['Absolute volumes per day', 'Indexed (weekday = 100)']):

    s = summary['sales'].values.astype(float)
    c = summary['calls'].values.astype(float)
    r = summary['reservations'].values.astype(float)

    if indexed:
        s = (s / s[0] * 100).round(1)
        c = (c / c[0] * 100).round(1)
        r = (r / r[0] * 100).round(1)

    b1 = ax.bar(x - w, s, w, label='Sales',        color=BLUE,  zorder=3)
    b2 = ax.bar(x,     c, w, label='Calls',         color=GRAY,  zorder=3)
    b3 = ax.bar(x + w, r, w, label='Reservations',  color=AMBER, zorder=3)

    for bars in [b1, b2, b3]:
        for bar in bars:
            h = bar.get_height()
            ax.text(bar.get_x() + bar.get_width() / 2, h + (8 if not indexed else 1.5),
                    f'{h:.0f}', ha='center', va='bottom', fontsize=8.5, color='#444')

    ax.set_xticks(x)
    ax.set_xticklabels(DAYS)
    ax.set_title(title)
    if indexed:
        ax.set_ylabel('Index (weekday = 100)')
        ax.axhline(100, color='#bbb', linewidth=1, linestyle='--', zorder=2)
    else:
        ax.set_ylabel('Count')
    ax.legend(frameon=False)

plt.suptitle('Sales vs calls vs reservations per day', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4 — Demand vs supply gap (unanswered calls)

In [ ]:
# May 2 call breakdown (from README: 302 of 435 had no van)
# With real data: calls['was_assigned'] = calls['shift_id'].notna()
served   = 133
no_van   = 302
total_calls = served + no_van

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# ── Donut: served vs unmet (May 2) ────────────────────────────────────────
wedges, texts, autotexts = ax1.pie(
    [no_van, served],
    labels=['No van\nassigned', 'Served'],
    colors=[RED, BLUE],
    autopct='%1.0f%%',
    startangle=90,
    wedgeprops={'width': 0.55, 'edgecolor': 'white', 'linewidth': 2},
    pctdistance=0.75,
)
for t in autotexts:
    t.set_fontsize(13)
    t.set_fontweight('bold')
    t.set_color('white')
ax1.text(0, 0, f'{total_calls}\ncalls', ha='center', va='center',
         fontsize=13, fontweight='bold', color='#333')
ax1.set_title('Calls without a van — 2 mei', fontweight='bold')

# ── Bar: calls per van per day ────────────────────────────────────────────
cpv = summary['calls_per_van'].values
bars = ax2.bar(DAYS, cpv, color=[BLUE, RED, AMBER], zorder=3, width=0.5)
for bar, v in zip(bars, cpv):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
             f'{v:.0f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax2.set_ylabel('Calls per active van')
ax2.set_title('Calls per van per day', fontweight='bold')
ax2.axhline(cpv[0], color='#aaa', linewidth=1, linestyle='--', zorder=2, label='Weekday baseline')
ax2.legend(frameon=False)

plt.suptitle('Demand vs supply gap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5 — Basket size (items per sale)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Grouped bar: sales vs sale_items ─────────────────────────────────────
ax = axes[0]
x  = np.arange(3)
w  = 0.35
b1 = ax.bar(x - w/2, summary['sales'],      w, label='Sales (tickets)', color=BLUE,  zorder=3)
b2 = ax.bar(x + w/2, summary['sale_items'], w, label='Items sold',       color=TEAL,  zorder=3)
for bars in [b1, b2]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 30, f'{h:,.0f}',
                ha='center', va='bottom', fontsize=8.5, color='#444')
ax.set_xticks(x)
ax.set_xticklabels(DAYS)
ax.set_ylabel('Count')
ax.set_title('Sales tickets vs items sold', fontweight='bold')
ax.legend(frameon=False)

# ── Line: basket size trend ────────────────────────────────────────────────
ax2 = axes[1]
bs  = summary['basket_size'].values
ax2.plot(DAYS, bs, marker='o', markersize=9, linewidth=2.5, color=BLUE, zorder=3)
ax2.fill_between(DAYS, bs, bs.min() - 0.2, alpha=0.08, color=BLUE)
for i, (d, v) in enumerate(zip(DAYS, bs)):
    ax2.annotate(f'{v:.1f}', (d, v), textcoords='offset points',
                 xytext=(0, 10), ha='center', fontsize=11, fontweight='bold', color=BLUE)
ax2.set_ylim(bs.min() - 0.5, bs.max() + 0.5)
ax2.set_ylabel('Items per sale')
ax2.set_title('Avg basket size per day', fontweight='bold')
ax2.axhline(bs[0], color='#aaa', linewidth=1, linestyle='--', label='Weekday baseline')
ax2.legend(frameon=False)

plt.suptitle('Basket size analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Saturday basket is {(bs[2]/bs[0]-1)*100:.0f}% larger than the weekday baseline.')

## 6 — Reservations per day & type

With real data, load `06_reservations.tsv` and group by `reservation_type`.

In [ ]:
# ── With real data ─────────────────────────────────────────────────────────
# res = pd.read_csv('foubertai_export/2026-05-02/06_reservations.tsv', sep='\t')
# res_counts = res.groupby('reservation_type').size()

# ── Placeholder from README ────────────────────────────────────────────────
# Actual type breakdown unknown yet — placeholder equal split for illustration
res_by_day = summary.set_index('day')['reservations']

# Placeholder type breakdown (replace once real data is loaded)
res_types = pd.DataFrame({
    'day':             ['30 apr (do)', '1 mei (feest)', '2 mei (za)'],
    'PRIVATE_RESERVATION':   [3, 5, 14],
    'EVENT_PAY_PER_PERSON':  [1, 4, 6],
    'EVENT_PAID_BY_HOST':    [1, 2, 2],
})
res_types = res_types.set_index('day')

colors_res = [BLUE, AMBER, GREEN]
fig, ax = plt.subplots(figsize=(10, 5))

bottom = np.zeros(3)
for col, color in zip(res_types.columns, colors_res):
    vals = res_types[col].values
    bars = ax.bar(DAYS, vals, bottom=bottom, label=col.replace('_', ' ').title(),
                  color=color, zorder=3, width=0.5)
    for bar, v in zip(bars, vals):
        if v > 0:
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_y() + bar.get_height()/2,
                    str(v), ha='center', va='center',
                    fontsize=9, color='white', fontweight='bold')
    bottom += vals

ax.set_ylabel('Number of reservations')
ax.set_title('Reservations per day by type\n(type breakdown is placeholder — load real data)', fontweight='bold')
ax.legend(frameon=False, loc='upper left')
plt.tight_layout()
plt.show()

## 7 — Hourly sales pattern (real data)

Uncomment once you have `02_sales.tsv` loaded.

In [ ]:
# ── Uncomment once real sales TSV files are loaded ─────────────────────────

# sales_all['hour'] = sales_all['datetime_start'].dt.hour
# sales_all['date_label'] = sales_all['datetime_start'].dt.date.astype(str)

# hourly = sales_all.groupby(['date_label', 'hour']).size().unstack(level=0).fillna(0)

# fig, ax = plt.subplots(figsize=(13, 5))
# colors_h = [GRAY, RED, BLUE]
# for col, color, label in zip(hourly.columns, colors_h, DAYS):
#     ax.plot(hourly.index, hourly[col], marker='o', markersize=5,
#             linewidth=2, color=color, label=label)
# ax.set_xlabel('Hour of day')
# ax.set_ylabel('Sales per hour')
# ax.set_xticks(range(8, 22))
# ax.legend(frameon=False)
# ax.set_title('Hourly sales pattern per day', fontweight='bold')
# plt.tight_layout()
# plt.show()

print('Load 02_sales.tsv files and uncomment this cell to see hourly patterns.')

## 8 — Feature importance (estimated signal strength for the model)

In [ ]:
features = pd.DataFrame({
    'feature': [
        'Day type (holiday / weekend / weekday)',
        'Call demand (app calls in zone)',
        'Reservation count in area',
        'Basket size (items / sale)',
        'Van availability (active vans)',
        'GPS velocity ≈ 0 (stop cluster)',
    ],
    'signal': [95, 82, 71, 58, 45, 38],
    'category': ['day', 'demand', 'demand', 'sales', 'supply', 'supply'],
})

cat_colors = {
    'day':    BLUE,
    'demand': RED,
    'sales':  AMBER,
    'supply': GREEN,
}

fig, ax = plt.subplots(figsize=(10, 5))
colors_f = [cat_colors[c] for c in features['category']]
bars = ax.barh(features['feature'], features['signal'], color=colors_f, zorder=3, height=0.55)

for bar, v in zip(bars, features['signal']):
    ax.text(v + 1, bar.get_y() + bar.get_height()/2,
            f'{v}%', va='center', fontsize=10, color='#444')

ax.set_xlim(0, 105)
ax.set_xlabel('Estimated signal strength')
ax.set_title('Key model features — ranked by signal strength', fontweight='bold')
ax.invert_yaxis()

legend_patches = [
    mpatches.Patch(color=BLUE,  label='Day type'),
    mpatches.Patch(color=RED,   label='Demand signal'),
    mpatches.Patch(color=AMBER, label='Sales behaviour'),
    mpatches.Patch(color=GREEN, label='Supply / GPS'),
]
ax.legend(handles=legend_patches, frameon=False, loc='lower right')
plt.tight_layout()
plt.show()

## 9 — GPS stop cluster (real data)

Load `gps/van_*.tsv` files, filter for low-velocity points, and map them.

In [ ]:
# ── Uncomment once GPS files are loaded ────────────────────────────────────

# import glob
# gps_files = glob.glob('foubertai_export/2026-05-02/gps/van_*.tsv')
# gps = pd.concat([pd.read_csv(f, sep='\t', parse_dates=['created_at']) for f in gps_files])

# stops = gps[gps['velocity'] < 2]   # velocity ≈ 0 = van is stopped

# fig, ax = plt.subplots(figsize=(10, 8))
# ax.scatter(stops['longitude'], stops['latitude'],
#            s=3, alpha=0.15, color=BLUE, linewidths=0)
# ax.set_xlabel('Longitude')
# ax.set_ylabel('Latitude')
# ax.set_title('GPS stop clusters (velocity < 2 km/h) — 2 mei 2026', fontweight='bold')
# plt.tight_layout()
# plt.show()

print('Load GPS TSV files and uncomment this cell to see stop clusters.')

## 10 — Sales scatter: location vs revenue (real data)

In [ ]:
# ── Uncomment once sales TSV is loaded ────────────────────────────────────

# fig, ax = plt.subplots(figsize=(10, 8))
# sc = ax.scatter(
#     sales_02['longitude_start'],
#     sales_02['latitude_start'],
#     c=sales_02['total_price_vati'],
#     cmap='YlOrRd',
#     s=40, alpha=0.7, linewidths=0
# )
# plt.colorbar(sc, ax=ax, label='Revenue per sale (incl. VAT)')
# ax.set_xlabel('Longitude')
# ax.set_ylabel('Latitude')
# ax.set_title('Sale locations coloured by revenue — 2 mei 2026', fontweight='bold')
# plt.tight_layout()
# plt.show()

print('Load 02_sales.tsv and uncomment this cell to see revenue by location.')